# Ingesta del archivo circuits.csv
1. Leer el archivo usando la API spark dataframe reader
1. Agregar las columnas de metadata 
    - Source File
    - Ingestion Timestamp
1. Write to bronze delta table    

In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/02.Ayuda_Bronze

In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/03.Configuracion_Ambiente

In [0]:
print("========================================")
print("PARAMETROS RECIBIDOS")
print("========================================")
print(f"storageName = {storageName}")
print(f"catalogo    = {catalogo}")
print(f"ruta        = {ruta}")
print("========================================")

archivo_fuente = f"{ruta}/circuits.csv"
nombre_tabla = f"{catalogo}.{esquema_bronze}.circuits"

print(f"archivo_fuente = {archivo_fuente}")
print(f"nombre_tabla   = {nombre_tabla}")

In [0]:
archivo_fuente = f"{ruta}/circuits.csv"
nombre_tabla = f"{catalogo}.{esquema_bronze}.circuits"

#### Step 1 - Read the CSV file using the dataframe reader API

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

esquema_circuits = StructType([
    StructField('circuitId',   StringType()),
    StructField("url",         StringType()),
    StructField("circuitName", StringType()),
    StructField("lat",         DoubleType()),
    StructField("long",        DoubleType()),
    StructField("locality",    StringType()),
    StructField("country",     StringType())
])

In [0]:
df_circuits = (
    spark.read
         .format('csv')
         .option('header', 'true')
         .option('mode', 'FAILFAST')
         .schema(esquema_circuits)
         .load(archivo_fuente)
)

In [0]:
display(df_circuits)

#### Step 2 - Add Metadata Columns
- Source File
- Ingestion Timestamp

In [0]:
df_final_circuits= add_ingesta_metadata(df_circuits)

In [0]:
display(df_final_circuits)

#### Step 3 - Write to bronze delta table

In [0]:
(
    df_final_circuits
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(nombre_tabla)
)

In [0]:
display(spark.table(nombre_tabla))

In [0]:
display(spark.table('formula1.bronze.circuits'))